# U-Net test - Fire Monitor - Brazil - Pantanal

**Campaign data:** `brazil / MONITOR_01` | **Sensor:** Sentinel-2 (10 m) | **Mosaic:** minnbr, monthly | **Region:** pantanal

Independent experiment: a **U-Net** trained on image chips, as an alternative to
the pixel MLP of the catalog (M4/M5). It reads the **same monthly mosaics**
(catalog M1/M2 COGs) and the **same monthly `fire`/`notFire` polygons** (catalog
M3 CSVs) from Cloud Storage, and imports nothing from the catalog versions.

Code: `network/catalog/experiments/unet_pantanal/` (branch `classification-unet`).
Guide: `README.md` in that folder.

**Use a fresh runtime with GPU** (Runtime > Change runtime type > GPU). Do not run
the catalog M4 trainer in the same runtime: it switches TensorFlow to v1 mode.

#### Steps

| Step | What it does | Output (GCS `.../MONITOR_01/UNET_01/`) |
|---|---|---|
| 0 | Setup, authentication, U-Net config | - |
| 1 | Inventory: mosaic grid and samples of each month | printed report |
| 2 | Spatial blocks + chips + labels (1 fire / 0 not fire / 255 unknown) | `chips/`, `splits/`, `reports/chips_*` |
| 3 | Normalization statistics (train chips only) | `stats/norm_stats.json` |
| 4 | Train one U-Net with the chips of every month in `PERIODS` | `models/` |
| 5 | Test metrics (threshold chosen on validation) | `reports/metrics_*.json` |
| 6 | Classify the region, one map per month in `PREDICT_PERIODS` | `predict/pantanal_{period}_prob.tif` |
| 7 | Compare with the MLP (A bands, B bands + indices, C U-Net) | `reports/comparison_*.json` |

## [0] Setup

In [ ]:
# 0.1 -- Colab setup (clone the branch, install, paths)
import sys, os, subprocess, shutil

BRANCH = "classification-unet"
REPO_DIR = "/content/brazil-fire"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/mapbiomas/brazil-fire.git", REPO_DIR], check=True)

EXP_DIR = os.path.join(REPO_DIR, "network/catalog/experiments/unet_pantanal")
subprocess.run(["pip", "install", "-q", "-r", os.path.join(EXP_DIR, "requirements.txt")], check=True)
sys.path.insert(0, os.path.join(EXP_DIR, "src"))

In [ ]:
# 0.2 -- U-Net configuration + authentication (every path comes from here)
import unet_env
from unet_config import make_config, paths

UCFG = make_config(
    # project (where the catalog data is)
    COUNTRY="brazil",
    BUCKET="mapbiomas-fire",
    GCS_PREFIX="sudamerica/brazil/CATALOG_01",
    CAMPAIGN="MONITOR_01",
    GEE_PROJECT="mapbiomas-brazil",
    REGIONS_ASSET="projects/mapbiomas-brazil/assets/FIRE/CATALOG_01/MONITOR_01/AUXILIARY/regiones_fuego_brazil_v1",
    # mosaics: LIBRARY_IMAGES/SENTINEL2/MONTHLY/MINNBR/{period}/COG/
    SENSOR="sentinel2",
    MOSAIC="minnbr",
    PERIODS=["2026_10"],             # months with mosaic AND samples (training); 'YYYY_MM'
    PREDICT_PERIODS=["2026_10"],     # months to classify (need only the mosaic)
    # experiment
    EXPERIMENT="UNET_01",            # GCS folder inside the campaign (outputs)
    MODEL_NAME="unet_monitor_pantanal_v1",
    REGION="pantanal",               # `region_nam` in the regions asset and `region` in the samples
    # First test: only green and swir2 are exported for 2026_10.
    BANDS=["green", "swir2"],
    # BANDS=["blue", "red", "nir", "swir1", "swir2"],   # full set, when exported
    INDICES=[],                      # no index can be computed from green + swir2
    # INDICES=["nbr", "nbr2", "mirbi", "ndvi", "bai", "evi"],   # needs the full set
    VALID_BAND=None,                 # "dayOfYear" when exported (valid mask + output band 2)
    SAMPLE_GLOB="*{period}*.csv",    # M3 CSVs in MONITOR_01/LIBRARY_SAMPLES; or SAMPLE_FILES=[...]
    CHIP=256, POINTS_PER_POLYGON=4, BLOCK_KM=25, SPLIT=(0.70, 0.15, 0.15),
    BATCH=8, EPOCHS=60, LR=1e-4, SEED=42,
    OVERLAP=16, INFER_TILE=512,
)
P = paths(UCFG)
unet_env.connect(UCFG)               # Colab pop-up; GEE with your account, GCS with the service account
print(f"Outputs: gs://{P['bucket']}/{P['root']}/")
print(f"Samples: gs://{P['bucket']}/{P['samples']}/")

## [1] Inventory (read only)
Checks that every band is on the same grid and summarizes the samples.

In [ ]:
import json
from unet_io import MosaicReader, load_samples, list_sample_files

for period in UCFG["PERIODS"]:
    reader = MosaicReader(UCFG, period)
    print(json.dumps(reader.inventory(), indent=2, default=str))
    print("Sample files:", list_sample_files(UCFG, period))
    samples, report = load_samples(UCFG, period, reader.crs)
    print(json.dumps({k: report[k] for k in ("periods_found", "period_corrected_rows", "by_class")}, indent=2))
    reader.close()

## [2] Chips and labels
Blocks of `BLOCK_KM` are split into train/val/test **before** the chips are made.
A chip that crosses blocks of different splits is dropped. Label: 1 fire, 0 not fire,
255 outside the polygons or nodata (ignored by the loss). Fire wins where polygons overlap.

In [ ]:
from IPython.display import Image, display
from unet_chips import build_chips
from unet_io import local_dir

for period in UCFG["PERIODS"]:
    summary = build_chips(UCFG, period, overwrite=False)
    if not summary.get("skipped"):
        print(json.dumps({k: summary[k] for k in ("blocks", "chips", "dropped", "fire_notfire_overlaps")},
                         indent=2, default=str)[:3000])
        png = os.path.join(local_dir("reports"), f"chips_preview_{period}.png")
        if os.path.exists(png):
            display(Image(png))

Check the preview: red = fire, blue = not fire. Look for chips over lagoons and wet areas (dark water looks like a burn scar) and over clouds or cloud shadows left in the monthly mosaic.

## [3] Normalization statistics (train chips only)

In [ ]:
from unet_chips import compute_stats
stats = compute_stats(UCFG)
for name, m, s in zip(stats["channels"], stats["mean"], stats["std"]):
    print(f"{name:6s} mean {m:9.4f} std {s:9.4f}")

## [4] Train the U-Net

In [ ]:
from unet_model import train
model, meta = train(UCFG)
print("epochs:", meta["epochs_run"], "| best val IoU:", max(meta["history"]["val_iou_fire"]))

## [5] Test metrics
Threshold with the best F1 on **validation**; the test is reported at that threshold and at 0.5.

In [ ]:
from unet_model import load_model
from unet_report import evaluate_unet
model = load_model(UCFG)
unet_results = evaluate_unet(UCFG, model)
print(json.dumps(unet_results["all"]["test_at_best"], indent=2))

## [6] Classify the whole region
Sliding window: each window reads `INFER_TILE + 2*OVERLAP` pixels and keeps only the
center, so there are no seams. Output in the M5 format (probability x 1000 + dayOfYear,
nodata -9999), same grid as the mosaic.

In [ ]:
from unet_io import read_json
from unet_predict import predict_mosaic, gee_upload_command
stats = read_json(P["stats"])
for period in UCFG["PREDICT_PERIODS"]:
    res = predict_mosaic(UCFG, period, model, stats)
    print(res)
    # Optional: upload to GEE (edit the asset path first)
    print(gee_upload_command(UCFG, period, f"projects/{UCFG['GEE_PROJECT']}/assets/FIRE/CATALOG_01/{UCFG['CAMPAIGN']}/{UCFG['EXPERIMENT']}/{UCFG['REGION']}_{period}_prob"))

## [7] Comparison with the MLP
Same test pixels for every model. A = MLP bands, B = MLP bands + indices (skipped when `INDICES` is empty), C = U-Net.

In [ ]:
from IPython.display import Markdown
from unet_report import compare_with_mlp
rows, table = compare_with_mlp(UCFG, unet_results)
display(Markdown(table))